# 🗺️ Cartographie Interactive — Mobilité Île-de-France

**Objectif** : produire des cartes interactives remettables aux élus et décideurs pour faciliter la prise de décision.

In [ ]:
import folium
import json
import pandas as pd
import numpy as np
from pathlib import Path

Path('outputs').mkdir(exist_ok=True)

# Coordonnées approximatives des communes
COORDS = {
    'Noisy-le-Grand': [48.848, 2.556],
    'Montreuil': [48.864, 2.448],
    'Vincennes': [48.848, 2.439],
    'Saint-Denis': [48.936, 2.357],
    'Nanterre': [48.892, 2.207],
    'Créteil': [48.781, 2.455],
    'Boulogne-Billancourt': [48.835, 2.240],
    'Versailles': [48.804, 2.130],
    'Argenteuil': [48.946, 2.247],
    'Évry': [48.632, 2.447],
    'Massy': [48.726, 2.272],
    'Cergy': [49.036, 2.063]
}

# Données communes (issues du notebook 02)
df = pd.DataFrame({
    'commune': list(COORDS.keys()),
    'validations_par_hab': [28.8, 41.3, 77.6, 53.9, 32.3, 31.2, 44.6, 43.0, 19.8, 30.0, 54.2, 23.1],
    'km_velo_par_1000hab': [0.25, 0.39, 0.57, 0.57, 0.32, 0.26, 0.64, 0.52, 0.13, 0.37, 0.73, 0.18],
    'profil': ['🟡','🟡','🟢','🔴','🟡','🔴','🟢','🟢','🔴','🔴','🟢','🟡'],
    'couleur': ['orange','orange','green','red','orange','red','green','green','red','red','green','orange']
})

print('Données prêtes pour la cartographie ✅')

## 1. Carte des profils territoriaux

In [ ]:
m = folium.Map(location=[48.85, 2.35], zoom_start=10,
               tiles='CartoDB positron')

# Légende
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 12px; border-radius: 8px;
     box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-size: 13px;">
  <b>Profil Mobilité</b><br>
  🟢 Bien desservie & aisée<br>
  🟡 Mobilité moyenne<br>
  🔴 Sous-équipée & précaire
</div>'''
m.get_root().html.add_child(folium.Element(legend_html))

for _, row in df.iterrows():
    lat, lon = COORDS[row['commune']]
    popup_text = f"""
    <b>{row['commune']}</b><br>
    Profil : {row['profil']}<br>
    Validations/hab : {row['validations_par_hab']:.1f}<br>
    Km vélo/1000 hab : {row['km_velo_par_1000hab']:.2f}
    """
    folium.CircleMarker(
        location=[lat, lon],
        radius=row['validations_par_hab'] / 3,
        color=row['couleur'], fill=True, fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=200),
        tooltip=row['commune']
    ).add_to(m)

m.save('outputs/carte_profils_mobilite.html')
print('✅ Carte sauvegardée : outputs/carte_profils_mobilite.html')
m

## 2. Carte chaleur — Pistes cyclables vs Usage TC

Cette carte met en évidence les communes où le rapport vélo/TC est le plus déséquilibré — zones prioritaires pour les investissements en mobilité douce.

In [ ]:
from folium.plugins import HeatMap

m2 = folium.Map(location=[48.85, 2.35], zoom_start=10, tiles='CartoDB dark_matter')

heat_data = [[COORDS[row['commune']][0], COORDS[row['commune']][1],
              row['validations_par_hab']] for _, row in df.iterrows()]

HeatMap(heat_data, min_opacity=0.4, radius=25, blur=20).add_to(m2)

# Marqueurs pistes cyclables insuffisantes
for _, row in df[df['km_velo_par_1000hab'] < 0.30].iterrows():
    lat, lon = COORDS[row['commune']]
    folium.Marker(
        [lat, lon],
        popup=f"⚠️ {row['commune']} — Vélo sous-développé",
        icon=folium.Icon(color='red', icon='warning-sign', prefix='glyphicon')
    ).add_to(m2)

m2.save('outputs/carte_chaleur_mobilite.html')
print('✅ Carte sauvegardée : outputs/carte_chaleur_mobilite.html')
m2

## ✅ Livrables produits

| Fichier | Description |
|---------|-------------|
| `outputs/carte_profils_mobilite.html` | Carte interactive des 3 profils territoriaux |
| `outputs/carte_chaleur_mobilite.html` | Carte de chaleur flux + zones prioritaires |
| `outputs/01_distribution_mode_commune.png` | Part modale par commune |
| `outputs/02_analyse_temporelle.png` | Courbe de charge journalière |
| `outputs/03_correlations_mobilite.png` | Matrice de corrélations |

**Ces livrables sont directement transmissibles aux élus et à la direction générale.**